# 06. Финальное решение (v6)

**LB 0,8628** (тег `v6`). Этот ноутбук даёт итоговый `answer.csv`.

Как устроено решение:
1. Пул ~1 400 кандидатов на запрос из восьми списков: BM25 по полям, BM25 с приоритетом близких объявлений, символьные n-граммы, вероятные микрокатегории рядом, микрокатегории похожих запросов train, эмбеддинги с учётом локации и без, эмбеддинги для размытых локаций.
2. 55 признаков пары «запрос / объявление»: текст, эмбеддинги, микрокатегория, локация, рейтинг, ранги кандидата в списках.
3. Два ранкера LightGBM (`lambdarank` и `binary`), итог: среднее мест кандидата по обоим.

Что нового относительно v5:
* эмбеддинги второго раунда из `03_embeddings` (Recall@100 0,391 → 0,420);
* из списков для размытых локаций оставлен только список по эмбеддингам, остальные ничего не добавляли;
* ансамбль двух ранкеров;
* пул бенчмарка строится после обучения, чтобы хватило памяти.

Нужен артефакт `artifacts/embeddings_v6`. Близость векторов считается точно в целых числах, поэтому md5 ответа одинаков на любой машине. Время выполнения: ~50 мин на CPU, ~16 ГБ RAM.

## Настройки

Единственная ячейка, которую может понадобиться поправить; `None` означает «определить автоматически». `EMB_NAME`: папка артефакта эмбеддингов. Проверка кода на маленьких выборках: `SMOKE_TEST = True`.

In [1]:
# ── Пути (None: автоматически) ──────────────────────────────────────────────────────────
REPO_DIR = None       # папка репозитория, если ноутбук открыт не из его папки notebooks/
DATA_DIR = None       # папка с train.parquet и benchmark_*.parquet; по умолчанию <репозиторий>/data
WORK_DIR = None       # куда писать отчёт и модель ранкера; по умолчанию <репозиторий>/artifacts
OUTPUT_DIR = None     # куда писать answer.csv; по умолчанию <репозиторий>/outputs
EMB_DIR = None        # папка артефакта из 03_embeddings; по умолчанию <WORK_DIR>/<EMB_NAME>
EMB_NAME = "embeddings_v6"   # артефакт второго раунда; "embeddings": вектора v4

# ── Режим ─────────────────────────────────────────────────────────────────────────────────
SMOKE_TEST = False    # True: маленькие выборки (проверка кода)

## 0. Окружение

Подключаем код из `src/`, ставим недостающие пакеты и закрепляем LightGBM 4.6.0: от версии зависит модель, а значит и `answer.csv`. Число потоков BLAS фиксируется до импорта numpy. Затем находим артефакт эмбеддингов.

In [ ]:
import base64, importlib, os, subprocess, sys
from pathlib import Path

for _name in ("REPO_DIR", "DATA_DIR", "WORK_DIR", "OUTPUT_DIR", "EMB_DIR", "HF_HOME", "HF_ENDPOINT"):
    if globals().get(_name):
        os.environ[_name] = str(globals()[_name])
for _name in ("DRY_RUN", "SMOKE_TEST"):
    if globals().get(_name):
        os.environ[_name] = "1"

N_THREADS = 4
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = str(N_THREADS)

# Код решения
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"            


def _github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def _git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def find_repo_root() -> Path:
    """REPO_DIR → папки выше текущей → (Kaggle или GITHUB_TOKEN) клонирование в текущую папку."""
    candidates = [Path(os.environ["REPO_DIR"]).expanduser()] if os.environ.get("REPO_DIR") else []
    candidates += [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "src" / "pipeline.py").exists():
            return path.resolve()
    token = _github_token()
    if not (token or Path("/kaggle/input").exists()):
        raise RuntimeError(
            "Не найден код решения (папка src/). Откройте ноутбук из папки notebooks/ клонированного "
            "репозитория или укажите путь к репозиторию в настройке REPO_DIR.")
    target = Path("/kaggle/working/avito-candgen") if Path("/kaggle/working").exists() else Path.cwd() / "avito-candgen"
    if not target.exists():
        _git("clone", "--quiet", REPO_URL, str(target), token=token)
    _git("-C", str(target), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(target), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    _git("-C", str(target), "checkout", "--quiet", "--force", "--detach",
         f"origin/{REPO_REF}" if is_branch else REPO_REF)
    return target


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip() or "(не git-репозиторий)"
except FileNotFoundError:
    COMMIT = "(git не установлен)"


def ensure_packages(requirements: dict) -> None:
    """requirements: модуль → pip-спецификации. Ставит только то, чего нет."""
    missing = []
    for module, specs in requirements.items():
        try:
            importlib.import_module(module)
        except ImportError:
            missing += specs
    if not missing:
        return
    print("устанавливаю:", " ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    if subprocess.run(cmd).returncode != 0 and subprocess.run(cmd + ["--user"]).returncode != 0:
        raise RuntimeError(f"Не удалось установить {missing}. Установите их вручную в терминале.")
    importlib.invalidate_caches()
    import site
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


ensure_packages({
    "pyarrow": ["pyarrow"],
    "pymorphy3": ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"],
})

from importlib import metadata
LIGHTGBM_VERSION = "4.6.0"
try:
    _lgb_installed = metadata.version("lightgbm")
except metadata.PackageNotFoundError:
    _lgb_installed = None
if _lgb_installed != LIGHTGBM_VERSION:
    if "lightgbm" in sys.modules:
        raise RuntimeError(f"В ядре уже загружен lightgbm {_lgb_installed} — перезапустите ядро (Kernel → Restart)")
    print(f"lightgbm: {_lgb_installed} → {LIGHTGBM_VERSION}")
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"lightgbm=={LIGHTGBM_VERSION}"]).returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user", f"lightgbm=={LIGHTGBM_VERSION}"],
                       check=True)
    importlib.invalidate_caches()
print(f"репозиторий: {REPO_ROOT}\nкоммит: {COMMIT}")

устанавливаю: pymorphy3==2.0.6 pymorphy3-dicts-ru==2.4.417150.4580142
lightgbm: 4.1.0 → 4.6.0
репозиторий: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev
коммит: 6d45af36e1ddf5e4e826dbdf563ddcc6c0d2b028


In [ ]:
import gc
import json

import numpy as np
import pandas as pd
from IPython.display import display

from src import analysis, eda
from src import encoder as enc
from src import knn_prior
from src.artifacts import find_embeddings_dir
from src.candidates import BASE_FEATURES, SOURCES, V5_FEATURES, V5_SOURCES, build_vocabs
from src.config import CFG, EMB_CFG, RANKER_V6
from src.data import load_benchmark, load_train, load_train_items_text
from src.paths import get_data_dir, get_output_dir, get_work_dir
from src.pipeline import STATS_COLS_V5, ItemLemmaCache, add_lemma_keys, build_index, build_pool
from src.ranker import (add_stage1, importance_table, rank_average, ranker_features, ranker_score,
                        sample_training_rows, train_ranker)
from src.ranking import PoolData, linear_score, pool_recall, predict_from_scores, weights_vector
from src.repro import file_md5, library_versions, seed_everything
from src.sampling import build_folds, build_validation, group_table, scheme_keys
from src.submit import save_answer, validate_answer
from src.text import Lemmatizer
from src.utils import memory_status, resources_report, timer
from src.validation import add_query_segments, mark_seen, per_query_recall, recall_at_k
from dataclasses import replace

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE_TEST") == "1"
R = RANKER_V6          
if SMOKE:
    R = replace(R, n_val_queries=300, fold_queries=300, max_rounds=150, early_stopping=30)
FEATURES = ranker_features(use_dense=True, v5=True)
DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
K, DEC = CFG.top_k, CFG.score_decimals
V2_W = weights_vector(BASE_FEATURES, R.v2_weights)
PREV = {"version": "v5", "lb": 0.854658, "answer_md5": "bf25d852bce82b30561275a4b987797d"}   
print(f"версия решения: {R.version}{' (SMOKE_TEST)' if SMOKE else ''}\ndata: {DATA_DIR}\nwork: {WORK_DIR}\nout:  {OUT_DIR}")
resources_report(WORK_DIR, need_ram_gb=16, need_disk_gb=3)
print(VERSIONS)

# --- артефакт эмбеддингов из 03_embeddings.ipynb ---
EMB_DIR = find_embeddings_dir(name=EMB_NAME)
EMB_MANIFEST = json.loads((EMB_DIR / "manifest.json").read_text())
print(f"\nэмбеддинги: {EMB_DIR}\n  модель {EMB_MANIFEST['model_name']} ({EMB_MANIFEST['mode']}), "
      f"объявлений {EMB_MANIFEST['n_items']:,}, запросов {EMB_MANIFEST['n_queries']:,}, "
      f"размерность {EMB_MANIFEST['dim']}, Recall@100 после дообучения {EMB_MANIFEST['recall@100_tuned']:.4f}")
if EMB_MANIFEST["mode"] != "full" and not SMOKE:
    print("[warn] артефакт собран в быстром режиме — для отправки нужен полный прогон 03_embeddings")

версия решения: v6
data: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/data
work: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts
out:  /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/outputs
RAM: 15.5 ГБ свободно из 16.0 | диск в /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts: 36 ГБ свободно | ядер CPU: 48
[warn] ноутбуку нужно около 16 ГБ RAM — возможна нехватка памяти
{'python': '3.10.13', 'numpy': '1.26.2', 'pandas': '2.0.3', 'scipy': '1.11.4', 'sklearn': '1.3.2', 'pyarrow': '14.0.1', 'pymorphy3': '2.0.6', 'lightgbm': '4.6.0', 'torch': '2.1.1+cu118'}

эмбеддинги: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts/embeddings_v6
  модель /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts/embeddings/model (full), объявлений 208,905, запросов 25,638, размерность 768, Recall@100 после дообучения 0.4200


## 1. Данные

Леммы и сегменты запросов, номер текста запроса у каждой строки train (для похожих запросов).

In [4]:
with timer("загрузка"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])
    TRAIN_TEXTS, train["qtext"] = knn_prior.train_text_codes(train)

OVERLAP = eda.overlap(train, bench_q, bench_items, CFG.item_stats_min_overlap)
print(f"\nгрупп-запросов: {len(groups):,}; из них все выбранные объявления в корпусе: "
      f"{groups['in_corpus'].sum():,} ({groups['in_corpus'].mean():.3f})")
print(f"уникальных текстов запросов (с фильтрами) в train: {len(TRAIN_TEXTS):,}")

[загрузка] 14.2 c
[подготовка запросов] 4.2 c

── Пересечение бенчмарка с train ─────────────────────────────────────────
объявлений корпуса, встречающихся в train: 0.096
запросов, чей текст встречается в train:  0.375
запросов, чей полный ключ есть в train:   0.044
→ статистики по item_id выключены (порог 0.5)

групп-запросов: 354,241; из них все выбранные объявления в корпусе: 18,415 (0.052)
уникальных текстов запросов (с фильтрами) в train: 107,957


## 2. Вектора запросов train

Вектора всех 108 тыс. текстов train для поиска похожих запросов. После `03` их в артефакте нет: при первом запуске их кодирует модель из артефакта (30 с на GPU), дальше они читаются из файлов.

In [5]:
def encode_train_texts(texts: list) -> np.ndarray:
    """Кодирует тексты запросов моделью из артефакта (так же, как 03 кодировал запросы для 04)."""
    try:
        import torch
    except ImportError as error:
        raise RuntimeError("Векторов запросов train ещё нет, а для их расчёта нужен PyTorch: запустите "
                           "ноутбук в ядре с torch (то же, где выполнялся 03_embeddings)") from error
    info = enc.device_info()
    amp = enc.choose_amp(EMB_CFG.amp_dtype, info)
    print(f"  устройство: {info['name']}, точность: {amp.name}")
    if info["device"] == "cpu":
        print("  [warn] GPU нет — на CPU кодирование займёт десятки минут")
    model = enc.BiEncoder.load(EMB_DIR / "model", info["device"], amp)
    emb = model.encode(texts, EMB_CFG.max_len_query, EMB_CFG.encode_batch, log_every=100)
    del model
    if info["device"] == "cuda":
        torch.cuda.empty_cache()
    return emb


with timer("вектора запросов train"):
    _emb = knn_prior.train_query_embeddings(
        EMB_DIR, TRAIN_TEXTS, encode_train_texts,
        meta={"model_name": EMB_MANIFEST["model_name"], "item_artifact_md5": EMB_MANIFEST["md5"],
              "commit": COMMIT, "versions": VERSIONS})
    TEXT_INDEX = knn_prior.TextIndex(_emb)
    del _emb
TRAIN_TEXTS_MANIFEST = json.loads((EMB_DIR / knn_prior.MANIFEST_FILE).read_text())
print(f"текстов: {TEXT_INDEX.n:,}; md5 дополнения: {TRAIN_TEXTS_MANIFEST['md5']}")
memory_status("после векторов запросов train")

кодирую 107,957 текстов запросов train (есть в артефакте: 0)
  устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb, точность: bf16
  закодировано 512 из 107,957
  закодировано 51,712 из 107,957
  закодировано 102,912 из 107,957
[вектора запросов train] 29.8 c
текстов: 107,957; md5 дополнения: {'train_queries.parquet': '86c91dca10d3e5312fbc02b69cb08c7e', 'train_query_embeddings.npy': '7843d189f2308e4b35338deb06b777a7'}
[память] после векторов запросов train: ноутбук занимает 5.3 ГБ, свободно 11.0 из 16.0 ГБ


## 3. Выборки валидации и фолдов

Те же, что в v4 и v5, запрос в запрос, поэтому числа сравнимы между версиями.

In [ ]:
ALL_ROWS = np.ones(len(train), dtype=bool)
KEYS = scheme_keys(groups)
with timer("выборки валидации и фолдов"):
    VAL = build_validation(train, groups, bench_q, R)
    FOLDS_ALL = {name: build_folds(train, groups, bench_q, R, VAL[name], KEYS[name]) for name in VAL}
print("запросов в фолдах:", {name: [len(f.queries) for f in folds] for name, folds in FOLDS_ALL.items()})

train = train[STATS_COLS_V5].copy()
memory_status("после выборок")

pd.concat({name: s.report.set_index(["текст", "страта"])["факт"] for name, s in VAL.items()},
          axis=1).assign(цель=VAL["injected"].report["цель"].to_numpy())

[warn] валидация in_corpus: в некоторых ячейках не хватило запросов — 2274 из 2500
[warn] фолд 0: в некоторых ячейках не хватило запросов — 3220 из 4000
[warn] фолд 1: в некоторых ячейках не хватило запросов — 2867 из 4000
[warn] фолд 2: в некоторых ячейках не хватило запросов — 2548 из 4000
[warn] фолд 3: в некоторых ячейках не хватило запросов — 2088 из 4000
[выборки валидации и фолдов] 26.0 c
запросов в фолдах: {'injected': [4000, 4000, 4000, 4000], 'in_corpus': [3220, 2867, 2548, 2088]}
[память] после выборок: ноутбук занимает 5.3 ГБ, свободно 10.9 из 16.0 ГБ


injected  in_corpus  цель
текст    страта                                                           
знакомый фильтр есть | локация обычная                443        443   443
         фильтр есть | локация только поисковая        84         84    84
         фильтра нет | локация обычная                333        333   333
         фильтра нет | локация только поисковая        78         78    78
новый    фильтр есть | локация обычная                336        336   336
         фильтр есть | локация только поисковая        61         61    61
         фильтра нет | локация обычная                954        782   954
         фильтра нет | локация только поисковая       211        157   211

## 4. Пулы и выбор схемы валидации

Пулы для обеих схем валидации. Похожие запросы ищутся только среди текстов статистик выборки. Пул бенчмарка строится в разделе 8, после обучения.

In [ ]:
vocabs = build_vocabs([train, bench_q], [train, bench_items])
cache = ItemLemmaCache(lem, CFG.desc_max_chars)
bench_ids = frozenset(bench_items["item_id"])


def items_with_injected(samples) -> pd.DataFrame:
    """Корпус бенчмарка + эталонные объявления выборок, которых в нём нет."""
    missing = sorted(dict.fromkeys(i for s in samples for rel in s.truth for i in rel if i not in bench_ids))
    extra = load_train_items_text(DATA_DIR, missing)
    print(f"подмешано объявлений: {len(extra):,}")
    return pd.concat([bench_items, extra[bench_items.columns]], ignore_index=True)


def make_index(name, items):
    """Индекс корпуса + вектора его объявлений из артефакта."""
    return build_index(name, items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R,
                       item_emb=enc.load_item_embeddings(EMB_DIR, items["item_id"]))


def make_pool(name, queries, truth, index, items, stats_mask):
    """Пул v5 с признаками и скором формулы v2 (stage1). Вектора запросов берутся из артефакта;
    докодирование — только запасной путь, как в v4 (для отправки так не делать)."""
    query_emb = enc.load_query_embeddings(EMB_DIR, enc.query_texts(queries), encode_train_texts)
    _, pool = build_pool(name, index, items, queries, train.loc[stats_mask], lem=lem,
                         vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"],
                         truth=truth, ext_cfg=R, query_emb=query_emb, text_index=TEXT_INDEX)
    return add_stage1(pool, index, R.v2_weights, DEC)


bench_index = make_index("корпус бенчмарка", bench_items)
inj_items = items_with_injected([VAL["injected"], *FOLDS_ALL["injected"]])
inj_index = make_index("корпус injected", inj_items)
CORPORA = {"injected": (inj_index, inj_items), "in_corpus": (bench_index, bench_items)}

VAL_POOLS = {name: make_pool(f"валидация {name}", s.queries, s.truth, *CORPORA[name], s.stats_mask)
             for name, s in VAL.items()}
cache.clear()          
memory_status("после пулов валидации")

[корпус бенчмарка: индекс корпуса] 136.1 c
подмешано объявлений: 19,693
[корпус injected: индекс корпуса] 56.0 c
  пул: 64/2500 запросов
  пул: 384/2500 запросов
  пул: 704/2500 запросов
  пул: 1024/2500 запросов
  пул: 1344/2500 запросов
  пул: 1664/2500 запросов
  пул: 1984/2500 запросов
  пул: 2304/2500 запросов
[валидация injected: статистики и пул] 125.9 c
валидация injected: запросов 2,500, строк пула 3,401,950 (~1361 на запрос)
  пул: 64/2274 запросов
  пул: 384/2274 запросов
  пул: 704/2274 запросов
  пул: 1024/2274 запросов
  пул: 1344/2274 запросов
  пул: 1664/2274 запросов
  пул: 1984/2274 запросов
  пул: 2274/2274 запросов
[валидация in_corpus: статистики и пул] 101.5 c
валидация in_corpus: запросов 2,274, строк пула 3,146,394 (~1384 на запрос)
[память] после пулов валидации: ноутбук занимает 11.6 ГБ, свободно 4.6 из 16.0 ГБ


Калибровка по LB, как в v3-v5: выбираем схему, где отправленная v2 ближе всего к своему LB.

In [8]:
calibration = []
for name, s in VAL.items():
    pool, (index, _) = VAL_POOLS[name], CORPORA[name]
    v2_pool = analysis.v2_rows(pool)
    recall_v2 = PoolData(v2_pool, index, BASE_FEATURES, s.n_rel).recall(V2_W, K, DEC)
    calibration.append({
        "схема": name, "запросов": len(s.queries), "Recall@50 v2": recall_v2,
        "LB v2": R.v2_lb, "расхождение": recall_v2 - R.v2_lb,
        "полнота пула v2": analysis.pool_hit_rate(v2_pool, s.n_rel).mean(),
        "полнота пула v6": analysis.pool_hit_rate(pool, s.n_rel).mean(),
    })
calibration = pd.DataFrame(calibration).set_index("схема").round(4)
SCHEME = calibration["расхождение"].abs().idxmin() if R.val_scheme == "auto" else R.val_scheme
display(calibration)
print(f"→ схема валидации: {SCHEME}")

,запросов,Recall@50 v2,LB v2,расхождение,полнота пула v2,полнота пула v6
схема,,,,,,
injected,2500,0.8597,0.8313,0.0284,0.9535,0.9850
in_corpus,2274,0.8761,0.8313,0.0448,0.9675,0.9912


→ схема валидации: injected


Выбрана `injected`. Полнота пула 0,985 (в v5 0,983).

**Вклад списков** и полнота пула по сегментам.

In [ ]:
V4_SOURCES = SOURCES + ["src_char_loc", "src_dense_loc"]


def pool_v4_rows(pool):
    return pool[pool[V4_SOURCES].any(axis=1)]


for name, s_ in VAL.items():
    pool = VAL_POOLS[name]
    present = [c for c in V4_SOURCES + V5_SOURCES if c in pool.columns]
    table = analysis.source_recall(pool, s_.n_rel)
    table.loc["пул без списков v5", "recall"] = analysis.pool_hit_rate(pool_v4_rows(pool), s_.n_rel).mean()
    for src in [c for c in V5_SOURCES if c in pool.columns]:    
        rest = [c for c in present if c != src]
        table.loc[f"пул без {src}", "recall"] = analysis.pool_hit_rate(pool[pool[rest].any(axis=1)], s_.n_rel).mean()
    print(f"\nвалидация {name}:")
    display(table.round(4))

s_, pool = VAL[SCHEME], VAL_POOLS[SCHEME]
seg = s_.queries[["seg_text", "seg_loc"]].assign(
    q_diffuse=pool.groupby("q")["q_diffuse"].first().reindex(range(len(s_.queries))).fillna(0).to_numpy() > 0,
    **{"без списков v5": analysis.pool_hit_rate(pool_v4_rows(pool), s_.n_rel),
       "пул v6": analysis.pool_hit_rate(pool, s_.n_rel)})
print(f"\nполнота пула по сегментам, валидация {SCHEME}:")
display(pd.concat({col: seg.groupby(col)[["без списков v5", "пул v6"]].mean().assign(запросов=seg.groupby(col).size())
                   for col in ["seg_text", "seg_loc", "q_diffuse"]}).round(4))


валидация injected:


,recall,avg_candidates
pool,0.9850,1360.78
src_char_loc,0.7752,1360.78
src_dense,0.5491,1360.78
src_dense_loc,0.8469,1360.78
src_dense_region,0.1600,1360.78
src_knn_loc,0.8629,1360.78
src_memo,0.0000,1360.78
src_prior_loc,0.8209,1360.78
src_text,0.6526,1360.78
src_text_loc,0.8764,1360.78



валидация in_corpus:


,recall,avg_candidates
pool,0.9912,1383.6385
src_char_loc,0.7842,1383.6385
src_dense,0.5577,1383.6385
src_dense_loc,0.8576,1383.6385
src_dense_region,0.1497,1383.6385
src_knn_loc,0.8816,1383.6385
src_memo,0.0000,1383.6385
src_prior_loc,0.8377,1383.6385
src_text,0.6961,1383.6385
src_text_loc,0.8999,1383.6385



полнота пула по сегментам, валидация injected:


без списков v5  пул v6  запросов
seg_text  знакомый                          0.9779  0.9892       938
          новый                             0.9616  0.9824      1562
seg_loc   локация обычная                   0.9890  0.9904      2066
          локация только поисковая          0.8664  0.9589       434
q_diffuse False                             0.9888  0.9903      2038
          True                              0.8745  0.9614       462

Для размытых локаций хватает одного списка по эмбеддингам: полнота пула у них 0,961 (в v5 было 0,947 с тремя списками). Без этого списка пул теряет 0,6 п.п.

**Точность микрокатегорий** по леммам и по похожим запросам, для запросов, чей эталон попал в пул.

In [10]:
pos = pool[pool["label"] == 1]
best = pos.groupby("q")[["prior_rank", "knn_rank"]].min()
best["seg_text"] = s_.queries["seg_text"].to_numpy()[best.index]
micro_table = best.groupby("seg_text").agg(
    запросов=("prior_rank", "size"),
    top1_леммы=("prior_rank", lambda r: (r < 1).mean()), top1_соседи=("knn_rank", lambda r: (r < 1).mean()),
    top3_леммы=("prior_rank", lambda r: (r < 3).mean()), top3_соседи=("knn_rank", lambda r: (r < 3).mean()))
display(micro_table.round(4))
knn_sim = pool.groupby("q")["q_knn_sim"].first()
print("близость лучшего соседа, медиана по сегментам:",
      knn_sim.groupby(s_.queries["seg_text"].to_numpy()[knn_sim.index]).median().round(4).to_dict())

,запросов,top1_леммы,top1_соседи,top3_леммы,top3_соседи
seg_text,,,,,
знакомый,929,0.8041,0.8170,0.9257,0.9257
новый,1535,0.4717,0.6782,0.6808,0.8267


близость лучшего соседа, медиана по сегментам: {'знакомый': 0.9998999834060669, 'новый': 0.949999988079071}


Для новых текстов похожие запросы ставят микрокатегорию эталона первой в 68% случаев, леммы в 47%.

## 5. Фолды для ранкера

Статистики каждого фолда считаются без его строк. Три фолда идут в обучение, четвёртый для ранней остановки.

In [ ]:
FOLDS = FOLDS_ALL[SCHEME]
fold_index, fold_items = CORPORA[SCHEME]

for name in [n for n in VAL_POOLS if n != SCHEME]:
    del VAL_POOLS[name]
if SCHEME == "in_corpus":
    del CORPORA["injected"], inj_index, inj_items
del FOLDS_ALL, pool, pos
memory_status("после выбора схемы")

[память] после выбора схемы: ноутбук занимает 11.4 ГБ, свободно 4.8 из 16.0 ГБ


In [12]:
TRAIN_COLS = ["gid", "label"] + FEATURES
VALID_COLS = ["q", "item", "label"] + FEATURES
train_parts = []
for f, fold in enumerate(FOLDS):
    pool = make_pool(f"фолд {f}", fold.queries, fold.truth, fold_index, fold_items, fold.stats_mask)
    if f < R.n_folds - 1:
        rows = sample_training_rows(pool, R, seed=CFG.seed + f)
        rows.insert(0, "gid", f * 1_000_000 + rows["q"].astype(np.int64))
        train_parts.append(rows[TRAIN_COLS])
        del rows
    else:
        VALID_POOL = pool[VALID_COLS]
    del pool
    gc.collect()

TRAIN_ROWS = pd.concat(train_parts, ignore_index=True)
del train_parts
print(f"обучающих строк: {len(TRAIN_ROWS):,} (позитивов {int(TRAIN_ROWS['label'].sum()):,}); "
      f"строк в фолде остановки: {len(VALID_POOL):,}")
memory_status("перед обучением ранкера")

  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 0: статистики и пул] 202.2 c
фолд 0: запросов 4,000, строк пула 5,459,725 (~1365 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 1: статистики и пул] 202.1 c
фолд 1: запросов 4,000, строк пула 5,465,637 (~1366 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 166

Перед обучением свободно 2,2 ГБ (в v5 было 0,7 ГБ).

## 6. Обучение ранкеров

`lambdarank` и `binary` с ранней остановкой по Recall@50 на четвёртом фолде. Затем ансамбль: среднее мест кандидата в запросе по обоим ранкерам. Что идёт в ответ, решает тот же фолд; валидация в выборе не участвует.

In [ ]:
valid_fold = FOLDS[-1]
valid_rank = fold_index.rank[VALID_POOL["item"].to_numpy()]
fold_scores = {"формула v2": PoolData(VALID_POOL, fold_index, BASE_FEATURES, valid_fold.n_rel).recall(V2_W, K, DEC)}
print(f"признаков у ранкера: {len(FEATURES)} (из них новых в v5: {len(V5_FEATURES)})")
MODELS = {}
for objective in R.objectives:
    with timer(f"LightGBM {objective}"):
        booster, best_iter, best = train_ranker(TRAIN_ROWS, VALID_POOL, valid_fold.n_rel, valid_rank,
                                                objective, R, CFG.seed, N_THREADS, K, DEC, features=FEATURES)
    MODELS[objective] = booster
    fold_scores[f"ранкер {objective}"] = best
    print(f"{objective}: лучшая итерация {best_iter}, Recall@{K} на фолде {best:.4f}")

OBJECTIVE = max(R.objectives, key=lambda o: fold_scores[f"ранкер {o}"])
RANKER = MODELS[OBJECTIVE]           


def final_score(pool, item_rank, choice):
    """Скор выбранного варианта: один ранкер или ансамбль всех (по местам в запросе)."""
    if choice == "ансамбль":
        return rank_average(pool, item_rank, [ranker_score(m, pool, N_THREADS, FEATURES) for m in MODELS.values()], DEC)
    return ranker_score(MODELS[choice], pool, N_THREADS, FEATURES)


if R.ensemble:
    fold_scores["ансамбль"] = pool_recall(VALID_POOL["q"].to_numpy(np.int64), valid_rank,
                                          VALID_POOL["label"].to_numpy(np.float64), valid_fold.n_rel,
                                          final_score(VALID_POOL, valid_rank, "ансамбль"), K, DEC)
CHOICE = "ансамбль" if R.ensemble and fold_scores["ансамбль"] > fold_scores[f"ранкер {OBJECTIVE}"] else OBJECTIVE
print(f"в ответ идёт: {CHOICE}")

del TRAIN_ROWS, VALID_POOL           
gc.collect()
memory_status("после обучения")
pd.Series(fold_scores, name=f"Recall@{K} на фолде остановки").round(4).to_frame()

признаков у ранкера: 55 (из них новых в v5: 12)
[100]	fold's recall@50: 0.915705
[LightGBM lambdarank] 90.4 c
lambdarank: лучшая итерация 9, Recall@50 на фолде 0.9196
[100]	fold's recall@50: 0.916423
[LightGBM binary] 80.6 c
binary: лучшая итерация 15, Recall@50 на фолде 0.9203
в ответ идёт: ансамбль
[память] после обучения: ноутбук занимает 11.6 ГБ, свободно 4.6 из 16.0 ГБ


,Recall@50 на фолде остановки
формула v2,0.8546
ранкер lambdarank,0.9196
ранкер binary,0.9203
ансамбль,0.9204


Оба ранкера остановились рано: 9 и 15 деревьев (в v5 было 172 и 149). Ранкер стал небольшой поправкой к формуле v2 и рангу по эмбеддингам, а дальше начинает подстраиваться под фолд. Ансамбль на фолде лучше обоих (0,9204), он и идёт в ответ.

## 7. Качество на валидации

Валидация не участвовала ни в обучении, ни в выборе модели. Рядом числа v5 на тех же запросах.

In [14]:
val_s, val_pool = VAL[SCHEME], VAL_POOLS[SCHEME]
val_index = CORPORA[SCHEME][0]
nq = len(val_s.queries)


def val_recall(pool, score):
    return recall_at_k(predict_from_scores(pool, val_index, score, K, DEC, nq), val_s.truth, K)


val_item_rank = val_index.rank[val_pool["item"].to_numpy()]
VAL_SCORES = {name: final_score(val_pool, val_item_rank, name)
              for name in list(MODELS) + (["ансамбль"] if R.ensemble else [])}
VAL_RESULTS = {"формула v2 на пуле v6": val_recall(val_pool, val_pool["stage1"].to_numpy(np.float64)),
               **{f"ранкер {name}" if name in MODELS else name: val_recall(val_pool, score)
                  for name, score in VAL_SCORES.items()}}
CHOICE_KEY = f"ранкер {CHOICE}" if CHOICE in MODELS else CHOICE
USE_RANKER = VAL_RESULTS[CHOICE_KEY] > VAL_RESULTS["формула v2 на пуле v6"]
ranker_val_score = VAL_SCORES[CHOICE]
POOL_RECALL = float(analysis.pool_hit_rate(val_pool, val_s.n_rel).mean())

compare = pd.DataFrame({"v6": {"полнота пула": POOL_RECALL, "Recall@50": VAL_RESULTS[CHOICE_KEY]}})
report_prev_path = WORK_DIR / "report_v5.json"
if report_prev_path.is_file():
    report_prev = json.loads(report_prev_path.read_text())
    if report_prev.get("val_scheme") == SCHEME:
        compare["v5"] = {"полнота пула": report_prev["pool_recall_val"],
                         "Recall@50": max(v for k, v in report_prev["val_recall@50"].items() if "ранкер" in k)}
        compare["прирост"] = compare["v6"] - compare["v5"]
    else:
        print(f"[warn] в report_v5.json другая схема валидации ({report_prev.get('val_scheme')}) — не сравниваю")
display(pd.Series(VAL_RESULTS, name=f"Recall@{K}, валидация {SCHEME}").round(4).to_frame())
display(compare.round(4))
print(f"в ответ идёт: {CHOICE if USE_RANKER else 'формула v2'}")

,"Recall@50, валидация injected"
формула v2 на пуле v6,0.8634
ранкер lambdarank,0.9198
ранкер binary,0.9231
ансамбль,0.9224


,v6,v5,прирост
Recall@50,0.9224,0.9154,0.0070
полнота пула,0.9850,0.9828,0.0022


в ответ идёт: ансамбль


In [15]:
final_val_score = ranker_val_score if USE_RANKER else val_pool["stage1"].to_numpy(np.float64)
val_pred = predict_from_scores(val_pool, val_index, final_val_score, K, DEC, nq)
r50 = per_query_recall(val_pred, val_s.truth, K)
val_queries = val_s.queries.assign(
    q_diffuse=np.where(val_pool.groupby("q")["q_diffuse"].first().reindex(range(nq)).fillna(0).to_numpy() > 0,
                       "размытая", "обычная"))
analysis.segment_table(val_queries, r50, analysis.pool_hit_rate(val_pool, val_s.n_rel),
                       seg_cols=("seg_text", "seg_filter", "seg_loc", "q_diffuse"))

n   share    pool  recall50
ось        сегмент                                                 
seg_text   знакомый                   938  0.3752  0.9892    0.9336
           новый                     1562  0.6248  0.9824    0.9157
seg_filter фильтр есть                924  0.3696  0.9902    0.9308
           фильтра нет               1576  0.6304  0.9819    0.9175
seg_loc    локация обычная           2066  0.8264  0.9904    0.9445
           локация только поисковая   434  0.1736  0.9589    0.8175
q_diffuse  обычная                   2038  0.8152  0.9903    0.9437
           размытая                   462  0.1848  0.9614    0.8285

In [16]:
importance = importance_table(RANKER)
display(importance.head(25))
print(f"важность — у ранкера {OBJECTIVE}; доля признаков v5: {importance.loc[importance['признак'].isin(V5_FEATURES), 'доля'].sum():.3f}")
errors = analysis.error_examples(val_pred, val_s.truth, val_s.queries, CORPORA[SCHEME][1],
                                 analysis.pool_hit_rate(val_pool, val_s.n_rel), n=15)
print(f"запросов без единого попадания: {int((r50 == 0).sum())} ({(r50 == 0).mean():.3f})")
errors

,признак,gain,доля
0,stage1_rank,646950.158680,0.603469
1,rank_dense_loc,194089.089550,0.181044
2,rank_knn_loc,36071.354763,0.033647
3,desc,26706.560078,0.024912
4,dense,26495.987362,0.024715
5,rank_region,21642.235077,0.020188
6,dist_rel,18319.882942,0.017089
7,rank_text_loc,14876.896915,0.013877
8,rank_dense,14016.123604,0.013074
9,stage1,7756.336342,0.007235


важность — у ранкера binary; доля признаков v5: 0.095
запросов без единого попадания: 180 (0.072)


,запрос,фильтры,та же локация,в пуле,заголовок эталона,параметры эталона
0,похудение,"Онлайн-запись Вид услуги Красота, здоровье",True,True,LPG массаж. Антицеллюлитный массаж,"Вид услуги Красота, здоровье Место оказания услуг Свердловская область, Екатеринбург, проспект К..."
1,фотомодель,Вид услуги Фото- и видеосъёмка,False,True,Модель на съемки,"Вид услуги Фото- и видеосъёмка Место оказания услуг Москва, Братиславская улица Тип стоимости за..."
2,тюнинг выхлопа,"Вид услуги Автосервис, аренда Тип услуги Автосервис",False,True,Установка управляемых выхлопов,"Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Республика Крым, Симфер..."
3,air touch,"Тип услуги Услуги парикмахера Вид услуги Красота, здоровье",False,True,Мелирование кератин аиртач стрижка,"Вид услуги Красота, здоровье Место оказания услуг Тюмень, улица Ленина, 81 Тип услуги Услуги пар..."
4,ремонт фанкойлов,,False,True,"Установка и Продажа Кондиционеров,Ремонт,Заправка","Вид услуги Монтаж и установка техники Место оказания услуг Санкт-Петербург, улица Дыбенко, 26 Ти..."
5,прокат мото,,False,False,Прокат снегоходов / квадроциклов / эндуро,"Вид услуги Праздники, мероприятия Место оказания услуг Республика Башкортостан, Абзелиловский ра..."
6,ремонт мебели,Вид услуги Ремонт и отделка,True,True,Муж на час Мастер на час,Вид услуги Ремонт и отделка Тип услуги Место оказания услуг Ростов-на-Дону Тип стоимости за услу...
7,курсы бровиста,,True,True,Мастер красоты,"Вид услуги Красота, здоровье Место оказания услуг пр-т Ленина, 70/1 Тип услуги Другое Тип стоимо..."
8,наращивание ногтей академический,"Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье",False,True,"Маникюр,педикюр,наращивание ногтей","Вид услуги Красота, здоровье Место оказания услуг Москва, Профсоюзная улица, 5/9 Тип услуги Мани..."
9,септик под ключ,"Вид услуги Сад, благоустройство",False,True,"Септик из бетонных колец, копка септиков колодцев","Вид услуги Сад, благоустройство Место оказания услуг Калужская область, Малоярославецкий район, ..."


**Итог:** Recall@50 0,922 (в v5 0,915), полнота пула 0,985. Выросли все сегменты: новые тексты 0,905 → 0,916, региональные локации 0,809 → 0,818, обычные 0,938 → 0,945. Второй по важности сигнал ранкера: ранг в списке по эмбеддингам (18%).

LB: 0,8628, +0,008 к v5.

## 8. Бенчмарк

Строим пул бенчмарка по статистикам всего train. Контрольная проверка: пайплайн без новых списков с весами v2 побайтно воспроизводит отправленный ответ v2. Файл проверяется на формат, прежний `answer.csv` сохраняется рядом.

In [ ]:
del VAL_POOLS, val_pool
gc.collect()
BENCH_POOL = make_pool("бенчмарк", bench_q, None, bench_index, bench_items, ALL_ROWS)
memory_status("после пула бенчмарка")

v2_bench = analysis.v2_rows(BENCH_POOL)
v2_pred = predict_from_scores(v2_bench, bench_index, linear_score(v2_bench[BASE_FEATURES].to_numpy(np.float64), V2_W),
                              K, DEC, len(bench_q))
v2_md5 = save_answer(bench_q["query_id"], v2_pred, WORK_DIR / "answer_v2_check.csv")
print(f"md5 ответа v2 из текущего пайплайна: {v2_md5} | отправленный: {R.v2_answer_md5} | "
      f"совпадает: {v2_md5 == R.v2_answer_md5}")


bench_item_rank = bench_index.rank[BENCH_POOL["item"].to_numpy()]
bench_score = (final_score(BENCH_POOL, bench_item_rank, CHOICE) if USE_RANKER
               else BENCH_POOL["stage1"].to_numpy(np.float64))
bench_pred = predict_from_scores(BENCH_POOL, bench_index, bench_score, K, DEC, len(bench_q))

ANSWER_PATH = OUT_DIR / "answer.csv"
prev_pred = None
if ANSWER_PATH.is_file() and file_md5(ANSWER_PATH) == PREV["answer_md5"]:
    prev_answer = pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False).set_index("query_id")
    prev_pred = [a.split() for a in prev_answer.loc[bench_q["query_id"].astype(str), "answer"]]
    (OUT_DIR / f"answer_{PREV['version']}.csv").write_bytes(ANSWER_PATH.read_bytes())   # прошлый ответ: рядом
save_answer(bench_q["query_id"], bench_pred, ANSWER_PATH)
CHECK = validate_answer(ANSWER_PATH, bench_q["query_id"], bench_items["item_id"], K)
print(CHECK)
overlap_v2 = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, v2_pred)])
print(f"совпадение с ответом v2: {overlap_v2:.3f} объявлений из топ-50 в среднем")
if prev_pred is not None:
    overlap_prev = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, prev_pred)])
    print(f"совпадение с ответом {PREV['version']}: {overlap_prev:.3f} (он сохранён как answer_{PREV['version']}.csv)")

  пул: 64/2452 запросов
  пул: 384/2452 запросов
  пул: 704/2452 запросов
  пул: 1024/2452 запросов
  пул: 1344/2452 запросов
  пул: 1664/2452 запросов
  пул: 1984/2452 запросов
  пул: 2304/2452 запросов
[бенчмарк: статистики и пул] 105.7 c
бенчмарк: запросов 2,452, строк пула 3,438,733 (~1402 на запрос)
[память] после пула бенчмарка: ноутбук занимает 11.4 ГБ, свободно 4.8 из 16.0 ГБ
md5 ответа v2 из текущего пайплайна: 2de61da58afd87fc244e225b7cccde58 | отправленный: 2de61da58afd87fc244e225b7cccde58 | совпадает: True
{'rows': 2452, 'min_items': 50, 'max_items': 50, 'md5': 'd1369e3e3a84bc839dd55231cf6cfac4'}
совпадение с ответом v2: 0.552 объявлений из топ-50 в среднем
совпадение с ответом v5: 0.668 (он сохранён как answer_v5.csv)


## 9. Артефакты

Модели обоих ранкеров, отчёт и md5 `answer.csv`. Повторный запуск даёт тот же md5: `d1369e3e3a84bc839dd55231cf6cfac4`.

In [18]:
for name, model in MODELS.items():
    model.save_model(str(WORK_DIR / f"ranker_{R.version}_{name}.txt"), num_iteration=model.best_iteration)
report = {
    "version": R.version,
    "answer_md5": CHECK["md5"],
    "val_scheme": SCHEME,
    "calibration": calibration.reset_index().to_dict(orient="records"),
    "v2_reproduced": v2_md5 == R.v2_answer_md5,
    "val_recall@50": VAL_RESULTS,
    "fold_recall@50": fold_scores,
    "choice": CHOICE,
    "best_iteration": {name: m.best_iteration for name, m in MODELS.items()},
    "use_ranker": bool(USE_RANKER),
    "pool_recall_val": POOL_RECALL,
    "micro_accuracy": micro_table.reset_index().to_dict(orient="records"),
    "config": CFG.as_dict(),
    "ranker_config": R.as_dict(),
    "embeddings": {k: v for k, v in EMB_MANIFEST.items() if k != "emb_config"},
    "train_queries": {k: TRAIN_TEXTS_MANIFEST[k] for k in ("n_texts", "md5")},
    "commit": COMMIT,
    "versions": VERSIONS,
}
(WORK_DIR / f"report_{R.version}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str))

for name, value in VAL_RESULTS.items():
    print(f"{name:28s} {value:.4f}")
print(f"полнота пула: {POOL_RECALL:.4f}")
print(f"answer.csv md5: {CHECK['md5']}")

формула v2 на пуле v6        0.8634
ранкер lambdarank            0.9198
ранкер binary                0.9231
ансамбль                     0.9224
полнота пула: 0.9850
answer.csv md5: d1369e3e3a84bc839dd55231cf6cfac4
